<a href="https://colab.research.google.com/github/VOIDxIIO/Colab-with-gui/blob/main/Colab-with-gui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🚀 One-Click Colab Desktop (RustDesk + Chrome + XFCE + Steam)
#@markdown Turn Google Colab into a full Linux desktop you can control from any device.
#@markdown ---
#@markdown **Set your RustDesk password (used to connect from the RustDesk app):**
RUSTDESK_PASSWORD = "colab1234" #@param {type:"string"}
#@markdown **Install Google Chrome?**
INSTALL_CHROME = True #@param {type:"boolean"}
#@markdown **Install Steam (GUI client)?**
INSTALL_STEAM = True #@param {type:"boolean"}
#@markdown ---

import subprocess, os, time, IPython
from google.colab import output

# ---------- helpers ----------
def run(cmd, quiet=False):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if not quiet and p.stdout.strip():
        print(p.stdout.strip())
    return p.returncode

def bg(cmd, log, env=None):
    subprocess.Popen(
        cmd, shell=True,
        stdout=open(log, "w"), stderr=subprocess.STDOUT,
        env={**os.environ, **(env or {})},
        start_new_session=True
    )

# ---------- 1. base packages ----------
print("📦 [1/8] Installing base packages (this takes ~2 min)...")
run("apt-get update -qq", quiet=True)
run("apt-get install -y -qq dbus-x11 xvfb xfce4 xfce4-goodies wget curl", quiet=True)

# ---------- 2. RustDesk ----------
print("🦀 [2/8] Installing RustDesk...")
run("wget -q https://github.com/rustdesk/rustdesk/releases/download/1.5.0/rustdesk-1.5.0-x86_64.deb -O /content/rustdesk.deb", quiet=True)
run("apt-get install -y -qq /content/rustdesk.deb", quiet=True)
run("touch /usr/lib/x86_64-linux-gnu/libcuda.so.1 && ldconfig 2>/dev/null", quiet=True)

# ---------- 3. Chrome (optional) ----------
if INSTALL_CHROME:
    print("🌐 [3/8] Installing Google Chrome...")
    run("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /content/chrome.deb", quiet=True)
    run("apt-get install -y -qq /content/chrome.deb", quiet=True)
    # patch launcher so it works as root
    run("sed -i 's|^Exec=/usr/bin/google-chrome-stable|Exec=/usr/bin/google-chrome-stable --no-sandbox --disable-dev-shm-usage --disable-gpu|' /usr/share/applications/google-chrome.desktop", quiet=True)
    run("update-desktop-database -q", quiet=True)

# ---------- 4. Steam (optional) ----------
if INSTALL_STEAM:
    print("🎮 [4/8] Installing Steam (this takes a few min)...")
    run("add-apt-repository -y multiverse", quiet=True)
    run("dpkg --add-architecture i386", quiet=True)
    run("apt-get update -qq", quiet=True)
    run("apt-get install -y -qq lib32gcc-s1 lib32stdc++6 libc6:i386 libstdc++6:i386", quiet=True)
    run("echo steam steam/question select 'I AGREE' | debconf-set-selections", quiet=True)
    run("echo steam steam/license note '' | debconf-set-selections", quiet=True)
    run("apt-get install -y -qq steam-installer", quiet=True)

# ---------- 5. cleanup ----------
print("🧹 [5/8] Cleaning up old processes...")
run("pkill -9 Xvfb; pkill -9 rustdesk; pkill -9 xfce4; pkill -9 chrome; sleep 2", quiet=True)

# ---------- 6. virtual display + desktop ----------
print("🖥  [6/8] Starting virtual display + XFCE desktop...")
bg("Xvfb :99 -screen 0 1280x720x24", "/content/xvfb.log")
time.sleep(3)
bg("dbus-daemon --system --fork", "/content/dbus.log")
time.sleep(2)

env = {"DISPLAY": ":99"}
bg("startxfce4", "/content/xfce.log", env)
time.sleep(10)

# ---------- 7. RustDesk server ----------
print("🔑 [7/8] Starting RustDesk and setting password...")
bg("rustdesk --server", "/content/rd_server.log", env)
time.sleep(8)
run(f"rustdesk --password {RUSTDESK_PASSWORD}", quiet=True)
bg("rustdesk --headless", "/content/rd_headless.log", env)
time.sleep(3)

# ---------- 8. keep-alive + output ----------
print("⏰ [8/8] Enabling keep-alive...")
display(IPython.display.Javascript('''
setInterval(function(){
  var b = document.querySelector("colab-connect-button");
  if (b) b.click();
  var ok = document.getElementById("ok");
  if (ok) ok.click();
}, 60000);
'''))

# grab the ID
id_result = subprocess.run("rustdesk --get-id", shell=True, capture_output=True, text=True)
rd_id = id_result.stdout.strip()

print("\n" + "="*55)
print("✅  YOUR COLAB DESKTOP IS READY")
print("="*55)
print(f"  RustDesk ID :  {rd_id}")
print(f"  Password    :  {RUSTDESK_PASSWORD}")
print("="*55)
print("\n📱 Next steps:")
print("  1. Install RustDesk from your phone/PC app store")
print("  2. Enter the ID above")
print("  3. Enter your password")
print("  4. You're in!\n")
print("⚠️  Keep this Colab tab open (minimized is fine).")
print("⚠️  Session resets in ~9–12 hours. Re-run this cell to rebuild.")

📦 [1/8] Installing base packages (this takes ~2 min)...
🦀 [2/8] Installing RustDesk...
🌐 [3/8] Installing Google Chrome...
🎮 [4/8] Installing Steam (this takes a few min)...
